# 실습 3 펭귄 종 분류 모델 구현하기 (학생용)

교과서 110~112쪽, 선택 활동 4(114~115쪽)

**문제 상황**: 남극의 급격한 기후 변화로 펭귄의 서식지가 위협받고 있지만, 펭귄 종을 구별하기 어려워 보호하기가 어렵다. 펭귄의 특징값으로 종(아델리, 턱끈, 젠투)을 분류하는 모델을 만들어 보자.

| 단계 | 하는 일 |
|---|---|
| 단계 1 문제 정의하기 | 펭귄 종을 분류하는 모델 구현하기 |
| 단계 2 데이터 수집 및 전처리하기 | penguins_size.csv, 결측치 제거, 특징과 타깃 정하기, 데이터 나누기 |
| 단계 3 모델 생성하기 | 결정트리 모델 학습, 트리 그리기 |
| 단계 4 모델 평가하기 | 분류 정확도, 혼동 행렬 |
| 선택 활동 4 | 로지스틱 회귀, k-최근접 이웃과 비교 |

**사용 방법**: 위에서부터 차례로 ▶ 버튼을 눌러 실행합니다. 오른쪽의 선택 상자와 슬라이더로 설정을 바꿔 다시 실행해 보세요.

In [ ]:
#@title 데이터 준비하기
how = "사이트에서 바로 받기" #@param ["사이트에서 바로 받기", "내 컴퓨터에서 올리기"]
#@markdown 디지털 교과서에 올려 둔 교과서 자료(penguins_size.csv)를 코랩으로 바로 가져옵니다. 직접 모은 파일을 쓰려면 '내 컴퓨터에서 올리기'를 고르세요.
if how.startswith('사이트'):
    import urllib.request
    for name in ['penguins_size.csv']:
        urllib.request.urlretrieve('https://aibook-dl.vercel.app/data/' + name, name)
        print('받았습니다:', name)
else:
    from google.colab import files
    uploaded = files.upload()


In [ ]:
#@title 단계 2 ① 데이터 불러오고 결측치 지우기 (교과서 110~111쪽)
import pandas as pd
df = pd.read_csv('penguins_size.csv')
print('결측치 수'); print(df.isnull().sum())
df.dropna(axis=0, inplace=True)            # 결측치가 있는 행 삭제
print('남은 데이터:', df.shape)

In [ ]:
#@title ② 특징과 타깃 정하고 데이터 나누기 (교과서 111쪽)
use_body_mass = False #@param {type:"boolean"}
#@markdown 체크하면 체질량(body_mass_g)도 특징에 넣습니다 (교과서 110쪽 지식 더하기).
features = ['culmen_length_mm', 'culmen_depth_mm', 'flipper_length_mm'] + (['body_mass_g'] if use_body_mass else [])
X = df[features]                           # 특징
y = df['species']                          # 타깃

from sklearn.model_selection import train_test_split
# stratify=y: 세 종을 훈련·테스트 데이터에 고르게 나눔 / random_state: 매번 같은 결과가 나오도록
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, stratify=y, random_state=42)
print('특징:', features)
print('훈련:', X_train.shape, ' 테스트:', X_test.shape)
print(y_train.value_counts())

In [ ]:
#@title 단계 3 결정트리 모델 학습하기 (교과서 112쪽)
max_depth = 0 #@param {type:"slider", min:0, max:8, step:1}
#@markdown 0이면 깊이 제한 없음(교과서와 같음). 1~8로 바꾸면 트리를 그 깊이까지만 키웁니다.
from sklearn.tree import DecisionTreeClassifier, plot_tree
import matplotlib.pyplot as plt
dt = DecisionTreeClassifier(max_depth=max_depth or None, random_state=42)
dt.fit(X_train, y_train)
print(f'훈련 데이터 분류 정확도: {dt.score(X_train, y_train):.3f}')
print(f'트리 깊이: {dt.get_depth()}, 단말 노드 수: {dt.get_n_leaves()}')

plt.figure(figsize=(20, 10))
plot_tree(dt, feature_names=X.columns, class_names=dt.classes_, max_depth=2, filled=True)
plt.show()

In [ ]:
#@title 단계 4 모델 평가하기: 분류 정확도와 혼동 행렬 (교과서 112쪽, 115쪽)
import seaborn as sns
from sklearn.metrics import confusion_matrix
dt_pred = dt.predict(X_test)
print(f'결정트리 테스트 정확도: {dt.score(X_test, y_test):.3f}')

conf = confusion_matrix(y_test, dt_pred, labels=dt.classes_)   # 행·열 순서 = dt.classes_
sns.heatmap(conf, annot=True, cmap='Blues', fmt='d', xticklabels=dt.classes_, yticklabels=dt.classes_)
plt.title('Penguin Classification (Decision Tree)')
plt.xlabel('Predicted'); plt.ylabel('Actual')
plt.show()

### 생각해 보기
- 훈련 데이터 정확도와 테스트 데이터 정확도를 비교해 보세요. 차이가 크다면 무엇을 의심할 수 있나요? (교과서 95쪽 과적합)
- `max_depth`를 2, 3, 4로 바꿔 보세요. 테스트 정확도는 어떻게 변하나요?
- 루트 노드(맨 위)에서 쓴 특징은 무엇인가요? 그 특징이 왜 처음에 쓰였을까요?

In [ ]:
#@title 선택 활동 4 ① 정규화하기 (교과서 114쪽)
from sklearn.preprocessing import MinMaxScaler
# 로지스틱 회귀와 k-최근접 이웃은 특징값의 범위에 영향을 받으므로 정규화한다.
# 훈련 데이터로 기준(최솟값·최댓값)을 정하고, 테스트 데이터는 그 기준으로 바꾸기만 한다.
scaler = MinMaxScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)
print('정규화된 훈련 데이터 범위:', X_train_s.min().round(2), '~', X_train_s.max().round(2))

In [ ]:
#@title ② 로지스틱 회귀와 k-최근접 이웃 학습하고 비교하기 (교과서 114~115쪽)
k = 5 #@param {type:"slider", min:1, max:25, step:1}
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
lr = LogisticRegression(max_iter=1000).fit(X_train_s, y_train)
knn = KNeighborsClassifier(n_neighbors=k).fit(X_train_s, y_train)
lr_pred, knn_pred = lr.predict(X_test_s), knn.predict(X_test_s)
print(f'결정트리 정확도     : {dt.score(X_test, y_test):.3f}')
print(f'로지스틱 회귀 정확도: {lr.score(X_test_s, y_test):.3f}')
print(f'k-최근접 이웃(k={k}) 정확도: {knn.score(X_test_s, y_test):.3f}')

model_to_show = "로지스틱 회귀" #@param ["로지스틱 회귀", "k-최근접 이웃", "결정트리"]
pred = {'로지스틱 회귀': lr_pred, 'k-최근접 이웃': knn_pred, '결정트리': dt_pred}[model_to_show]
conf = confusion_matrix(y_test, pred, labels=dt.classes_)
sns.heatmap(conf, annot=True, cmap='Blues', fmt='d', xticklabels=dt.classes_, yticklabels=dt.classes_)
plt.title('Penguin Classification'); plt.xlabel('Predicted'); plt.ylabel('Actual')
plt.show()

In [ ]:
#@title ③ 정밀도와 재현율 살펴보기 (교과서 115쪽 지식 더하기)
from sklearn.metrics import classification_report
print(classification_report(y_test, pred, digits=3))

### 정리하기
1. 세 모델 중 가장 정확한 모델은 무엇인가요? 혼동 행렬에서 어떤 종을 가장 많이 헷갈렸나요?
2. 정규화를 하지 않고 k-최근접 이웃을 학습하면 결과가 어떻게 달라질까요? (교사용 노트북에서 확인)